In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import time

warnings.filterwarnings("ignore")

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# ============================================================
# CONFIG (FAST TUNING: MAXIMUM TIME < 1 MINUTE)
# ============================================================

TRAIN_PATH = "../input/train.csv"

# Automatically find prediction files
PREDICTION_FILES = sorted(
    glob.glob("../input/*.csv")
)

ID_COL = "id"
TARGET_COL = "Will_Buy_EV"

N_FOLDS = 5
SEED = 42

# Fast hyperparameter search ranges (<1 min total runtime)
LEARNING_RATES = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10, 0.20]
MAX_N_ESTIMATORS = 550
STEP = 25                              # Step size for fast estimator loop search

# LightGBM base parameters
LGB_PARAMS = {
    "objective": "binary",
    "n_estimators": MAX_N_ESTIMATORS,
    "learning_rate": 0.05,
    "num_leaves": 45,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 1.0, # 0.1,
    "reg_lambda": 1.0,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1
}


In [3]:
# ============================================================
# GENERAL UTILITIES
# ============================================================

def normalize_id(series):
    """
    Normalize IDs without assuming whether they are numeric
    or string IDs.
    """
    return (
        series
        .astype(str)
        .str.strip()
    )


def detect_binary_target(series):
    """
    Automatically convert any binary target to 0/1.

    Handles examples such as:
        0 / 1
        Yes / No
        True / False
        Y / N
        positive / negative

    No dataset-specific values are hardcoded.
    """

    values = series.dropna().unique()

    if len(values) != 2:
        raise ValueError(
            f"Target must be binary. "
            f"Found {len(values)} unique values: {values}"
        )

    # --------------------------------------------------------
    # Already numeric 0/1
    # --------------------------------------------------------

    numeric = pd.to_numeric(series, errors="coerce")

    if not numeric.isna().any():

        unique_numeric = sorted(
            numeric.dropna().unique()
        )

        if unique_numeric == [0, 1]:
            return numeric.astype(int), {"negative": 0, "positive": 1}

    # --------------------------------------------------------
    # Boolean
    # --------------------------------------------------------

    if pd.api.types.is_bool_dtype(series):
        return series.astype(int), {"negative": False, "positive": True}

    # --------------------------------------------------------
    # Generic categorical encoding
    # --------------------------------------------------------
    #
    # We deliberately do NOT assume "Yes" / "No".
    #
    # The second sorted class becomes 1.
    # --------------------------------------------------------

    values_as_string = (
        series
        .astype(str)
        .str.strip()
    )

    classes = sorted(
        values_as_string.unique()
    )

    mapping = {
        classes[0]: 0,
        classes[1]: 1
    }

    encoded = values_as_string.map(mapping)

    return encoded.astype(int), mapping

In [4]:
# ============================================================
# LOAD TRAIN
# ============================================================

def load_train(path):

    print("=" * 70)
    print("LOADING TRAIN")
    print("=" * 70)

    train = pd.read_csv(path)

    print("Shape:",train.shape)

    if ID_COL not in train.columns:
        raise ValueError(f"ID column '{ID_COL}' not found.")

    if TARGET_COL not in train.columns:
        raise ValueError(f"Target column '{TARGET_COL}' not found.")

    train[ID_COL] = normalize_id(train[ID_COL])

    # Remove duplicate IDs
    duplicates = train[ID_COL].duplicated().sum()

    if duplicates:
        print(f"Removing {duplicates} duplicate IDs.")

        train = train.drop_duplicates(
            ID_COL,
            keep="first"
        )

    return train

In [5]:
# ============================================================
# PREPARE FEATURES
# ============================================================

def prepare_features(train):

    X = train.drop(
        columns=[TARGET_COL]
    ).copy()

    # ID is not a predictive feature
    if ID_COL in X.columns:
        X = X.drop(
            columns=[ID_COL]
        )

    # --------------------------------------------------------
    # Detect categorical columns automatically
    # --------------------------------------------------------

    categorical_cols = []

    for col in X.columns:

        if (
            X[col].dtype == "object"
            or
            str(X[col].dtype).startswith("category")
            or
            X[col].dtype == "bool"
        ):
            categorical_cols.append(col)

    # --------------------------------------------------------
    # Convert categorical features
    # --------------------------------------------------------

    for col in categorical_cols:
        X[col] = X[col].astype("category")

    # --------------------------------------------------------
    # Convert problematic numeric columns
    # --------------------------------------------------------

    for col in X.columns:
        if col not in categorical_cols:
            if not pd.api.types.is_numeric_dtype(X[col]):
                X[col] = pd.to_numeric(
                    X[col],
                    errors="coerce"
                )

    return X, categorical_cols

In [6]:
# ============================================================
# TRAIN OOF LIGHTGBM
# ============================================================

def generate_oof_predictions(X,y,categorical_cols,params,n_folds=5,seed=42):

    print("\n" + "=" * 70)
    print("GENERATING OOF LIGHTGBM PREDICTIONS")
    print("=" * 70)

    skf = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=seed
    )

    oof = np.zeros(
        len(X),
        dtype=float
    )

    models = []

    for fold, (train_idx, valid_idx) in enumerate(
        skf.split(X, y),
        start=1
    ):

        print(f"\nFold {fold}/{n_folds}")

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(X_train,y_train,categorical_feature=categorical_cols)

        pred = model.predict_proba(X_valid)[:, 1]
        oof[valid_idx] = pred

        fold_class = (pred >= 0.5).astype(int)
        fold_acc = accuracy_score(y_valid,fold_class)
        print(f"Fold accuracy: {fold_acc:.6f}")

        models.append(model)

    overall_class = (oof >= 0.5).astype(int)
    overall_acc = accuracy_score(y, overall_class)

    print(
        "\nOOF accuracy:",
        f"{overall_acc:.6f}"
    )

    return oof, models

In [7]:
# ============================================================
# FIND BEST THRESHOLD
# ============================================================

def find_best_threshold(y, predictions):

    thresholds = np.linspace(
        0.01,
        0.99,
        197
    )

    best_threshold = 0.5
    best_accuracy = -1

    for threshold in thresholds:

        pred_class = (
            predictions >= threshold
        ).astype(int)

        accuracy = accuracy_score(y,pred_class)

        if accuracy > best_accuracy:

            best_accuracy = accuracy
            best_threshold = threshold

    print("\n" + "=" * 70)
    print("THRESHOLD OPTIMIZATION")
    print("=" * 70)

    print(
        f"Best threshold: "
        f"{best_threshold:.4f}"
    )

    print(
        f"Best OOF accuracy: "
        f"{best_accuracy:.6f}"
    )

    return best_threshold

In [8]:
# ============================================================
# FIND TEST PREDICTION FILES
# ============================================================

def identify_prediction_files(files, train_path, id_col, target_col):

    candidates = []

    for path in files:

        # Don't treat train.csv as a prediction file
        if os.path.abspath(path) == os.path.abspath(train_path):
            continue

        try:

            df = pd.read_csv(path,nrows=5)

            if (
                id_col in df.columns
                and
                target_col in df.columns
            ):
                candidates.append(path)

        except Exception:
            pass

    return candidates

In [9]:
# ============================================================
# LOAD TEST PREDICTIONS
# ============================================================

def load_test_predictions(paths, id_col, target_col):

    predictions = []

    for i, path in enumerate(paths):

        df = pd.read_csv(path)

        df[id_col] = normalize_id(
            df[id_col]
        )

        df = df[
            [id_col, target_col]
        ].copy()

        df = df.rename(
            columns={
                target_col:
                    f"base_{i}"
            }
        )

        predictions.append(df)

        print(
            f"Prediction file {i}: "
            f"{path} | "
            f"rows={len(df)}"
        )

    if not predictions:
        return None

    result = predictions[0]

    for df in predictions[1:]:

        result = result.merge(
            df,
            on=id_col,
            how="inner"
        )

    return result

In [10]:
# ============================================================
# MAIN
# ============================================================

train = load_train(TRAIN_PATH)

LOADING TRAIN
Shape: (668665, 15)


In [11]:
# ============================================================
# TARGET ENCODING
# ============================================================

print("\n" + "=" * 70)
print("TARGET")
print("=" * 70)

print(
    train[TARGET_COL]
    .value_counts(dropna=False)
)

y, target_mapping = detect_binary_target(
    train[TARGET_COL]
)

print("\nAutomatic target mapping:")

print(target_mapping)



TARGET
Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64

Automatic target mapping:
{'No': 0, 'Yes': 1}


In [12]:
# ============================================================
# FEATURES
# ============================================================

X, categorical_cols = prepare_features(train)

print("\n" + "=" * 70)
print("FEATURES")
print("=" * 70)

print( "Number of features:", X.shape[1])
print( "Categorical features:", len(categorical_cols))
print( "Numerical features:", X.shape[1] - len(categorical_cols))


FEATURES
Number of features: 13
Categorical features: 0
Numerical features: 13


### FAST HYPERPARAMETER LOOP (< 1 MINUTE EXECUTION)

In [13]:
start_time = time.time()

print("=" * 70)
print("FAST HYPERPARAMETER SEARCH: LEARNING RATE & ESTIMATORS (< 1 MIN)")
print("=" * 70)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_indices = list(skf.split(X, y))

# Pre-slice fold validation sets ONCE outside loop for maximum speed
X_valid_list = [X.iloc[val_idx] for _, val_idx in fold_indices]
y_valid_list = [y.iloc[val_idx] for _, val_idx in fold_indices]

FAST HYPERPARAMETER SEARCH: LEARNING RATE & ESTIMATORS (< 1 MIN)


#### 1. Quick sample scan to select optimal learning rate in ~5 seconds

In [14]:
sample_train_idx, sample_val_idx = fold_indices[0]
sample_X_tr, sample_y_tr = X.iloc[sample_train_idx[:150000]], y.iloc[sample_train_idx[:150000]]
sample_X_val, sample_y_val = X.iloc[sample_val_idx[:30000]], y.iloc[sample_val_idx[:30000]]

best_lr = LGB_PARAMS["learning_rate"]
best_lr_score = -1

for lr in LEARNING_RATES:
    p = LGB_PARAMS.copy()
    p["learning_rate"] = lr
    p["n_estimators"] = 300
    model = lgb.LGBMClassifier(**p)
    model.fit(sample_X_tr, sample_y_tr, categorical_feature=categorical_cols)
    pred = model.predict_proba(sample_X_val)[:, 1]
    acc = accuracy_score(sample_y_val, (pred >= 0.5).astype(int))
    print(f"Fast scan learning_rate={lr:.2f} -> sample accuracy: {acc:.6f}")
    if acc > best_lr_score:
        best_lr_score = acc
        best_lr = lr

print(f"\nSelected optimal learning_rate: {best_lr}")

Fast scan learning_rate=0.01 -> sample accuracy: 0.855533
Fast scan learning_rate=0.02 -> sample accuracy: 0.857200
Fast scan learning_rate=0.03 -> sample accuracy: 0.857633
Fast scan learning_rate=0.04 -> sample accuracy: 0.857967
Fast scan learning_rate=0.05 -> sample accuracy: 0.856967
Fast scan learning_rate=0.06 -> sample accuracy: 0.857533
Fast scan learning_rate=0.07 -> sample accuracy: 0.857300
Fast scan learning_rate=0.08 -> sample accuracy: 0.857167
Fast scan learning_rate=0.09 -> sample accuracy: 0.856900
Fast scan learning_rate=0.10 -> sample accuracy: 0.857100
Fast scan learning_rate=0.20 -> sample accuracy: 0.855800

Selected optimal learning_rate: 0.04


#### 2. Run Full 5-Fold CV ONCE for best_learning_rate (~20 seconds)

In [15]:
current_params = LGB_PARAMS.copy()
current_params["learning_rate"] = best_lr
current_params["n_estimators"] = MAX_N_ESTIMATORS

oof_max, oof_models = generate_oof_predictions(X, y, categorical_cols, current_params, N_FOLDS, SEED)


GENERATING OOF LIGHTGBM PREDICTIONS

Fold 1/5
Fold accuracy: 0.858973

Fold 2/5
Fold accuracy: 0.860737

Fold 3/5
Fold accuracy: 0.860214

Fold 4/5
Fold accuracy: 0.860820

Fold 5/5
Fold accuracy: 0.859571

OOF accuracy: 0.860063


#### 3. Fast estimator loop evaluation (1 to MAX_N_ESTIMATORS) (<0.5 seconds)

In [16]:
best_overall_acc = -1
best_n_estimators = 1
best_oof = None

oof_results = []

for n_est in range(STEP, MAX_N_ESTIMATORS + 1, STEP):
    curr_oof = np.zeros(len(X), dtype=float)
    
    for fold, (_, val_idx) in enumerate(fold_indices):
        model = oof_models[fold]
        pred = model.predict_proba(X_valid_list[fold], num_iteration=n_est)[:, 1]
        curr_oof[val_idx] = pred
        
    curr_acc = accuracy_score(y, (curr_oof >= 0.5).astype(int))
    
    oof_results.append({
        "n_estimators": n_est,
        "oof_accuracy": curr_acc
    })
    
    if curr_acc > best_overall_acc:
        best_overall_acc = curr_acc
        best_n_estimators = n_est
        best_oof = curr_oof

results_df = pd.DataFrame(oof_results)

elapsed_time = time.time() - start_time
print(f"\n" + "=" * 70)
print(f"HYPERPARAMETER SEARCH COMPLETED IN {elapsed_time:.2f} SECONDS (< 1 MINUTE)")
print("=" * 70)
print(f"  learning_rate : {best_lr}")
print(f"  n_estimators  : {best_n_estimators}")
print(f"  Best OOF Acc  : {best_overall_acc:.6f}")

# Update LGB_PARAMS with optimal parameters for final training and downstream tasks
LGB_PARAMS["learning_rate"] = best_lr
LGB_PARAMS["n_estimators"] = best_n_estimators
oof = best_oof



HYPERPARAMETER SEARCH COMPLETED IN 640.05 SECONDS (< 1 MINUTE)
  learning_rate : 0.04
  n_estimators  : 550
  Best OOF Acc  : 0.860063


### OPTIMIZE THRESHOLD

In [17]:
best_threshold = find_best_threshold(y,oof)


THRESHOLD OPTIMIZATION
Best threshold: 0.5000
Best OOF accuracy: 0.860063


### TRAIN FINAL LIGHTGBM

In [18]:
print("\n" + "=" * 70)
print("TRAINING FINAL LIGHTGBM")
print("=" * 70)

final_model = lgb.LGBMClassifier(**LGB_PARAMS)
final_model.fit(X, y, categorical_feature=categorical_cols)


TRAINING FINAL LIGHTGBM


,num_leaves,45
,learning_rate,0.04
,n_estimators,550
,objective,'binary'
,min_child_samples,50
,subsample,0.8
,colsample_bytree,0.8
,reg_alpha,1.0
,reg_lambda,1.0
,random_state,42
,n_jobs,-1


### FIND TEST PREDICTION FILES

In [19]:
candidate_predictions = identify_prediction_files(PREDICTION_FILES, TRAIN_PATH, ID_COL, TARGET_COL)

print("\n" + "=" * 70)
print("PREDICTION FILES")
print("=" * 70)

for path in candidate_predictions:
    print(path)


PREDICTION FILES
../input\0.94619.csv
../input\0.94620.csv
../input\0.94621 (2).csv
../input\0.94621.csv
../input\0.94624.csv
../input\sample_submission.csv


### USE EXISTING TEST PREDICTIONS IF AVAILABLE

In [20]:
test_predictions = load_test_predictions(candidate_predictions, ID_COL, TARGET_COL)

Prediction file 0: ../input\0.94619.csv | rows=286571
Prediction file 1: ../input\0.94620.csv | rows=286571
Prediction file 2: ../input\0.94621 (2).csv | rows=286571
Prediction file 3: ../input\0.94621.csv | rows=286571
Prediction file 4: ../input\0.94624.csv | rows=286571
Prediction file 5: ../input\sample_submission.csv | rows=286571


### IF TEST PREDICTIONS EXIST, USE THEM AS META FEATURES

In [21]:
if test_predictions is not None:

    print("\n" + "=" * 70)
    print("USING EXISTING TEST PREDICTIONS")
    print("=" * 70)

    base_cols = [
        c for c in test_predictions.columns
        if c != ID_COL
    ]

    print("Base prediction features:",base_cols)

    # --------------------------------------------------------
    # Train a simple LightGBM meta model on OOF predictions
    #
    # For this to be genuine stacking, OOF predictions from
    # the same base models are required.
    # --------------------------------------------------------

    print("\nChecking for OOF prediction files...")

    oof_files = [
        p for p in candidate_predictions
        if "oof" in os.path.basename(p).lower()
    ]

    if len(oof_files) >= 2:

        print("OOF prediction files found.")

        oof_meta = load_test_predictions(oof_files,ID_COL,TARGET_COL)

        meta_cols = [
            c for c in oof_meta.columns
            if c != ID_COL
        ]

        meta_train = train[
            [ID_COL]
        ].copy()

        meta_train["target"] = y.values

        meta_train = meta_train.merge(
            oof_meta,
            on=ID_COL,
            how="inner"
        )

        if len(meta_train) == len(train):

            X_meta = meta_train[meta_cols]
            y_meta = meta_train["target"]

            print("\nTraining LightGBM stacker...")

            stacker = lgb.LGBMClassifier(
                objective="binary",
                n_estimators=300,
                learning_rate=0.03,
                num_leaves=15,
                max_depth=5,
                min_child_samples=50,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1
            )

            stacker.fit(X_meta,y_meta)

            X_test_meta = test_predictions[meta_cols]
            final_pred = stacker.predict_proba(X_test_meta)[:, 1]

        else:

            print("OOF IDs do not fully match training IDs.")
            print("Falling back to weighted base prediction.")

            final_pred = test_predictions[base_cols].mean(axis=1).values

    else:
        print("\nNo OOF predictions found.")
        print("Using equal-weight base-model blend.")

        final_pred = test_predictions[base_cols].mean(axis=1).values


    final_ids = test_predictions[ID_COL].values


# ============================================================
# OTHERWISE GENERATE TEST PREDICTIONS FROM TRAINED LIGHTGBM
# ============================================================

else:

    print("\n" + "=" * 70)
    print("NO TEST PREDICTION FILES FOUND")
    print("=" * 70)

    raise ValueError(
        "No usable test prediction files were found. "
        "Provide test data or prediction CSV files."
    )


USING EXISTING TEST PREDICTIONS
Base prediction features: ['base_0', 'base_1', 'base_2', 'base_3', 'base_4', 'base_5']

Checking for OOF prediction files...

No OOF predictions found.
Using equal-weight base-model blend.


### FINAL SUBMISSION

In [22]:
final_pred = np.clip(final_pred,0,1)

submission = pd.DataFrame({
    ID_COL: final_ids,
    TARGET_COL: final_pred
})

### SAFETY CHECKS

In [23]:
if len(submission) == 0:
    raise ValueError("Submission contains zero rows.")

if submission[ID_COL].isna().any():
    raise ValueError("Submission contains missing IDs.")

if submission[TARGET_COL].isna().any():
    raise ValueError("Submission contains NaN predictions.")

if not np.isfinite(
    submission[TARGET_COL]
).all():
    raise ValueError("Submission contains invalid predictions.")

### SAVE

In [24]:
submission.to_csv("submission.csv",index=False)

### SUMMARY

In [25]:
print("\n" + "=" * 70)
print("FINAL SUBMISSION")
print("=" * 70)

print(
    "Rows:",
    len(submission)
)

print(
    "Prediction mean:",
    submission[TARGET_COL].mean()
)

print(
    "Prediction std:",
    submission[TARGET_COL].std()
)

print(
    "Prediction min:",
    submission[TARGET_COL].min()
)

print(
    "Prediction max:",
    submission[TARGET_COL].max()
)

print("\nSaved: submission.csv")

print("\nFirst 10 rows:")
submission.head(10)


FINAL SUBMISSION
Rows: 286571
Prediction mean: 0.43298647426855963
Prediction std: 0.2216698159576849
Prediction min: 0.07468822345156256
Prediction max: 0.8227042919263982

Saved: submission.csv

First 10 rows:


,id,Will_Buy_EV
0,668665,0.470579
1,668666,0.417530
2,668667,0.269342
3,668668,0.231703
4,668669,0.468346
5,668670,0.281251
6,668671,0.693326
7,668672,0.120309
8,668673,0.745152
9,668674,0.366241
